# Bipower Variation and Jumps

Realized variance counts a price jump in full, while bipower variation, the scaled sum of products of adjacent absolute returns, leaves it out ([Barndorff-Nielsen and Shephard (2004)](https://doi.org/10.1093/jjfinec/nbh001)). This tutorial adds a jump to the simulated day, measures it with the gap between the two, and tests it with the statistic of [Barndorff-Nielsen and Shephard (2006)](https://doi.org/10.1093/jjfinec/nbi022).

In [ ]:
Pollis.packages("HighFrequencyCovariance", "Plots", "SpecialFunctions")

## A Day With a Jump

The simulated day of the first tutorial, with the price falling 1% at once at tick 14,000, as after a news announcement. The true quadratic variation is then the continuous 0.01^2 plus the squared jump, 0.01^2.

In [ ]:
using HighFrequencyCovariance, Dates, Random, SpecialFunctions, Statistics
import Plots

rng = Xoshiro(7)
n = 23400  # a tick every second over a trading day of 6.5 hours
efficient = log(100) .+ cumsum(0.01 / sqrt(n) .* randn(rng, n))  # efficient log price, daily volatility 1%
without_jump = copy(efficient)
efficient[14000:end] .-= 0.01  # a jump of -1%
noise = 0.0002 .* randn(rng, n)  # microstructure noise of 2 basis points
ticks = (time = collect(1.0:n), asset = fill(:SPX, n), logprice = efficient .+ noise)
ts = SortedDataFrame(HighFrequencyCovariance.DataFrame(ticks), :time, :asset, :logprice, Second(1))
span = duration(ts; in_dates_period = false)  # seconds from the first tick to the last

function grid_returns(t, p, every)  # log returns between the last prices at or before each multiple of every seconds
	grid = range(first(t), last(t); step = every)
	return diff([p[searchsortedlast(t, g)] for g in grid])
end

In [ ]:
r = grid_returns(ticks.time, ticks.logprice, 300)  # five-minute returns
Plots.bar(100 .* r; xlabel = "five-minute interval", ylabel = "return (%)", label = "", title = "Five-minute returns: one stands out")

## Realized Variance Against Bipower Variation

Realized variance squares every return, so the jump counts in full. Bipower variation multiplies each absolute return by its neighbour, and the factor pi / 2 makes the products comparable with squared returns. The jump is multiplied only by ordinary returns, so it adds little.

In [ ]:
RV = sum(r .^ 2)
BV = (pi / 2) * sum(abs.(r[2:end]) .* abs.(r[1:end-1]))
(realized = sqrt(RV), bipower = sqrt(BV), true_with_jump = sqrt(0.01^2 + 0.01^2), true_continuous = 0.01)

## The Barndorff-Nielsen and Shephard Test

With no jumps, realized variance and bipower variation estimate the same thing, and their difference divided by its standard error is roughly standard normal. The standard error uses tripower quarticity, built from products of three adjacent absolute returns to the power 4/3. Jumps raise only realized variance, so the test is one-sided.

In [ ]:
function bns_test(r)
	n = length(r)
	RV = sum(r .^ 2)
	BV = (pi / 2) * sum(abs.(r[2:end]) .* abs.(r[1:end-1]))
	mu43 = 2^(2 / 3) * gamma(7 / 6) / gamma(1 / 2)  # E|Z|^(4/3) for a standard normal Z
	TQ = n * mu43^(-3) * sum(abs.(r[3:end]) .^ (4 / 3) .* abs.(r[2:end-1]) .^ (4 / 3) .* abs.(r[1:end-2]) .^ (4 / 3))
	Z = (RV - BV) / sqrt(((pi / 2)^2 + pi - 5) * TQ / n)
	return (Z = Z, p_value = 0.5 * erfc(Z / sqrt(2)), jump_share = max(RV - BV, 0) / RV)
end
bns_test(r)

## The Same Day Without the Jump

The efficient price before the jump was added, with the same noise. Realized variance and bipower variation now agree up to sampling error, and the test does not reject.

In [ ]:
r0 = grid_returns(ticks.time, without_jump .+ noise, 300)
(realized = sqrt(sum(r0 .^ 2)), bipower = sqrt((pi / 2) * sum(abs.(r0[2:end]) .* abs.(r0[1:end-1]))), test = bns_test(r0))

## Two-Scales Volatility Includes the Jump

Two-scales volatility corrects for noise, not for jumps: like realized volatility, it measures the whole quadratic variation, jump included.

In [ ]:
(two_scales = two_scales_volatility(ts, [:SPX])[1][:SPX] * sqrt(span), bipower = sqrt(BV), true_with_jump = sqrt(0.01^2 + 0.01^2))

## Bipower Variation Needs Sparse Returns

Bipower variation does not correct for noise. On every tick, products of adjacent noisy returns are dominated by the noise, and bipower variation is far above the truth, as realized variance is.

In [ ]:
[(seconds = k, bipower = round(sqrt((pi / 2) * sum(abs.(grid_returns(ticks.time, ticks.logprice, k)[2:end]) .* abs.(grid_returns(ticks.time, ticks.logprice, k)[1:end-1]))), sigdigits = 4)) for k in [1, 10, 60, 300, 900]]

## Things to Try

- Make the jump smaller, 0.3%, and see when the test stops finding it at five minutes.
- Put two jumps in adjacent five-minute intervals: they multiply each other and enter bipower variation.
- Run **Jump Share of the Variance** in High-Frequency Volatility on your own ticks.